Define un wrapper personalizado para el entorno CarRacing, que analiza los colores del camino y detecta movimiento para modificar la recompensa del agente según la fase, junto con un callback que registra y muestra las recompensas por episodio durante el entrenamiento.

In [1]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
import time
import matplotlib.pyplot as plt

class CarRacingRewardWrapper(gym.Wrapper):
    def __init__(self, env, fase=1):
        super().__init__(env)
        self.fase_actual = fase
        self.reset_estadisticas()
        self.episode_count = 0
        
    def reset_estadisticas(self):
        self.pasos_en_verde = 0
        self.pasos_en_gris = 0
        self.ultima_posicion = None
        self.steps_sin_movimiento = 0
        self.total_steps = 0
        self.acciones_positivas = 0
        self.acciones_negativas = 0
        
    def analizar_colores(self, obs):
        try:
            region = obs[70:85, 38:58]
            
            verde_count = 0
            gris_count = 0
            total = max(region.shape[0] * region.shape[1], 1)
            
            for i in range(region.shape[0]):
                for j in range(region.shape[1]):
                    r, g, b = region[i, j]
                    r, g, b = int(r), int(g), int(b)
                    
                    diff_rg = abs(r - g)
                    diff_gb = abs(g - b)
                    if diff_rg < 25 and diff_gb < 25 and 90 < r < 120:
                        gris_count += 1
                    elif g > r + 20 and g > b + 20 and g > 95:
                        verde_count += 1
            
            return gris_count / total, verde_count / total
        except:
            return 0.0, 0.0
    
    def detectar_movimiento(self, obs):
        if self.ultima_posicion is None:
            self.ultima_posicion = obs.copy()
            return True
            
        try:
            diff = np.mean(np.abs(obs - self.ultima_posicion))
            self.ultima_posicion = obs.copy()
            return diff > 1.5
        except:
            return True
    
    def step(self, action):
        try:
            obs, reward, terminated, truncated, info = self.env.step(action)
            self.total_steps += 1
            
            gris_pct, verde_pct = self.analizar_colores(obs)
            se_mueve = self.detectar_movimiento(obs)
            
            custom_reward = 0.0
            
            # SISTEMA DE RECOMPENSAS POR FASE
            if self.fase_actual == 1:
                # FASE 1: RECOMPENSAS SIMPLES Y GENEROSAS
                if se_mueve:
                    custom_reward += 1.0  # Generosa por movimiento
                    self.steps_sin_movimiento = 0
                else:
                    self.steps_sin_movimiento += 1
                    if self.steps_sin_movimiento > 10:
                        custom_reward -= 0.5
                
                if gris_pct > 0.3:  # Umbral bajo
                    custom_reward += 2.0  # Generosa por camino
                    self.pasos_en_gris += 1
                elif verde_pct > 0.25:
                    custom_reward -= 1.0  # Penalización suave
                    
                # Cualquier aceleración es buena en Fase 1
                if action[1] > 0.1:
                    custom_reward += 0.3
                    
            else:
                # FASE 2: RECOMPENSAS MÁS EXIGENTES
                if se_mueve:
                    custom_reward += 0.8  # Menos generosa
                    self.steps_sin_movimiento = 0
                else:
                    self.steps_sin_movimiento += 1
                    if self.steps_sin_movimiento > 8:  # Más exigente
                        custom_reward -= 0.8
                
                if gris_pct > 0.4:  # Umbral más alto
                    custom_reward += 1.5  # Menos generosa
                    self.pasos_en_gris += 1
                    
                    # Bonus por aceleración ÓPTIMA (no cualquier aceleración)
                    if 0.3 < action[1] < 0.7:
                        custom_reward += 0.8
                        
                elif verde_pct > 0.2:  # Más sensible al verde
                    custom_reward -= 1.2  # Penalización más fuerte
                
                # Recompensa por giros SUAVES (no cualquier giro)
                if 0.1 < abs(action[0]) < 0.5:
                    custom_reward += 0.4
            
            # LOGROS (comunes a ambas fases pero con diferentes umbrales)
            if self.fase_actual == 1:
                if self.pasos_en_gris >= 10:  # Fácil en Fase 1
                    custom_reward += 2.0
                    print(f"🏅 FASE 1: ¡Buen comienzo! {self.pasos_en_gris} pasos en gris")
                    self.pasos_en_gris = 0
            else:
                if self.pasos_en_gris >= 20:  # Más difícil en Fase 2
                    custom_reward += 3.0
                    print(f"🏅 FASE 2: ¡Conducción avanzada! {self.pasos_en_gris} pasos en gris")
                    self.pasos_en_gris = 0
            
            # LIMITAR recompensas
            custom_reward = np.clip(custom_reward, -3.0, 3.0).item()
            
            # Mostrar progreso
            if self.total_steps % 100 == 0:
                print(f"📊 Fase {self.fase_actual} - Paso {self.total_steps}: Reward={custom_reward:.1f}")
            
            return obs, float(custom_reward), bool(terminated), bool(truncated), info
            
        except Exception as e:
            print(f"❌ Error en step: {e}")
            return np.zeros((96, 96, 3)), -1.0, True, True, {}
    
    def reset(self, **kwargs):
        self.episode_count += 1
        print(f"\n🔄 FASE {self.fase_actual} - EPISODIO {self.episode_count}")
        self.reset_estadisticas()
        try:
            return self.env.reset(**kwargs)
        except:
            return np.zeros((96, 96, 3)), {}

class VisualizadorCallback(BaseCallback):
    def __init__(self, fase=1, verbose=0):
        super().__init__(verbose)
        self.fase = fase
        self.episodio = 0
        self.recompensa_episodio = 0
        self.mejor_recompensa = -float('inf')
        self.recompensas_por_episodio = []  # Guardar todas las recompensas
    
    def _on_step(self):
        try:
            reward = self.locals['rewards'][0]
            if not np.isnan(reward):
                self.recompensa_episodio += reward
            
            if self.locals['dones'][0]:
                self.episodio += 1
                self.recompensas_por_episodio.append(self.recompensa_episodio)  # Guardar recompensa
                
                if self.recompensa_episodio > self.mejor_recompensa:
                    self.mejor_recompensa = self.recompensa_episodio
                    print(f"🎉 FASE {self.fase} - EPISODIO {self.episodio} - NUEVO RÉCORD: {self.recompensa_episodio:.1f}")
                else:
                    print(f"📊 FASE {self.fase} - EPISODIO {self.episodio} - Recompensa: {self.recompensa_episodio:.1f}")
                
                self.recompensa_episodio = 0
            
            return True
        except:
            return True


Esta función entrena el modelo PPO en la Fase 1 del entorno CarRacing, usando recompensas generosas para fomentar movimiento y recorrer el camino, registra las recompensas por episodio con un callback, guarda el modelo entrenado y devuelve tanto el modelo como el callback.

In [2]:
def entrenar_fase_1():
    print("🚀 INICIANDO FASE 1 - EXPLORACIÓN Y APRENDIZAJE BÁSICO")
    
    env = gym.make("CarRacing-v3", render_mode="human", continuous=True)
    env = CarRacingRewardWrapper(env, fase=1)
    
    model = PPO(
        "MlpPolicy",
        env,
        verbose=1,
        learning_rate=2.5e-4,
        n_steps=2048,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.8,
        policy_kwargs=dict(net_arch=[128, 128])
    )
    
    callback = VisualizadorCallback(fase=1)
    model.learn(total_timesteps=10000, callback=callback)
    
    model.save("carracing_fase1")
    print("✅ FASE 1 COMPLETADA - Modelo guardado")
    print(f"🏆 Mejor recompensa obtenida: {callback.mejor_recompensa:.1f}")
    
    env.close()
    return model, callback

Esta función entrena el modelo PPO en la Fase 2 del entorno CarRacing, utilizando recompensas más exigentes para refinar y optimizar el comportamiento del coche, puede cargar los pesos aprendidos en la Fase 1, registra las recompensas por episodio con un callback, guarda el modelo final y devuelve tanto el modelo como el callback.

In [ ]:
def entrenar_fase_2(modelo_fase1=None):
    print("🚀 INICIANDO FASE 2 - REFINAMIENTO")
    env = gym.make("CarRacing-v3", render_mode="human", continuous=True)
    env = CarRacingRewardWrapper(env, fase=2)
    
    # ⚙️ CONFIGURACIÓN FASE 2
    model = PPO(
        "MlpPolicy",
        env,
        verbose=1,
        learning_rate=1e-4,      # Más lento para ajustes finos
        n_steps=1024,            # Menos exploración
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.15,
        ent_coef=0.005,          # MENOS exploración
        vf_coef=0.5,
        max_grad_norm=0.6,
        policy_kwargs=dict(
            net_arch=[128, 128]  # Red más grande para patrones complejos
        )
    )
    
    # CARGAR PESOS DE FASE 1 SI EXISTE
    if modelo_fase1 is not None:
        print("🔄 Cargando conocimientos de FASE 1...")
        model.set_parameters(modelo_fase1.get_parameters())
    
    callback = VisualizadorCallback(fase=2)
    print("🏁 COMIENZA FASE 2...")
    model.learn(total_timesteps=15000, callback=callback)
    
    model.save("carracing_fase2")
    print("✅ FASE 2 COMPLETADA - Modelo guardado")
    print(f"🏆 Mejor recompensa obtenida: {callback.mejor_recompensa:.1f}")
    
    env.close()
    return model, callback

Esta función prueba un modelo entrenado en el entorno CarRacing ejecutando 3 episodios de manera determinista, muestra la interacción del coche en pantalla, acumula la recompensa total y los pasos por episodio, y finalmente cierra el entorno.

In [4]:
def probar_modelo(fase=2):
        print(f"🎮 PROBANDO MODELO FASE {fase}...")
    
        env = gym.make("CarRacing-v3", render_mode="human", continuous=True)
        env = CarRacingRewardWrapper(env, fase=fase)
    
        model = PPO.load("carracing_fase1" if fase==1 else "carracing_fase2")
            
        print(f"✅ Modelo FASE {fase} cargado")
        
        for episodio in range(3):
            obs, info = env.reset()
            done = False
            total_reward = 0
            pasos = 0
            
            print(f"🏁 EPISODIO PRUEBA {episodio + 1}")
            
            while not done and pasos < 500:
                action, _ = model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = env.step(action)
                total_reward += reward
                pasos += 1
                done = terminated or truncated
                
                time.sleep(0.02)
            
            print(f"🏁 EPISODIO {episodio+1} - Pasos: {pasos} | Recompensa: {total_reward:.1f}")
        
        env.close()

Esta sección es el programa principal, que al ejecutarse entrena primero la Fase 1 (exploración básica) y luego la Fase 2 (refinamiento), guardando los modelos entrenados y los callbacks asociados para cada fase.

In [ ]:
# 🎮 PROGRAMA PRINCIPAL
if __name__ == "__main__":
    # ENTRENAMIENTO COMPLETO
    modelo_fase1, callback1 = entrenar_fase_1()
    modelo_fase2, callback2 = entrenar_fase_2(modelo_fase1)

🚀 INICIANDO FASE 1 - EXPLORACIÓN Y APRENDIZAJE BÁSICO
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.

🔄 FASE 1 - EPISODIO 1
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
📊 Fase 1 - Paso 100: Reward=2.0
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo! 10 pasos en gris
🏅 FASE 1: ¡Buen comienzo!

Esta parte genera las gráficas de aprendizaje para cada fase: muestra la recompensa acumulada por episodio y marca con un punto rojo el episodio donde se obtuvo la mejor recompensa para la Fase 1 (callback1) y la Fase 2 (callback2).

In [ ]:
# 📊 GRAFICA DE APRENDIZAJE
plt.figure(figsize=(10,5))
plt.plot(callback1.recompensas_por_episodio, label="Recompensa por episodio")
mejor_ep = callback1.recompensas_por_episodio.index(callback1.mejor_recompensa)
plt.scatter(mejor_ep, callback1.mejor_recompensa, color='red', label="Mejor recompensa", zorder=5)
plt.xlabel("Episodio")
plt.ylabel("Recompensa acumulada")
plt.title("Aprendizaje Fase 1")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# 📊 GRAFICA DE APRENDIZAJE FASE 2
plt.figure(figsize=(10,5))
plt.plot(callback2.recompensas_por_episodio, label="Recompensa por episodio")
mejor_ep = callback2.recompensas_por_episodio.index(callback2.mejor_recompensa)
plt.scatter(mejor_ep, callback2.mejor_recompensa, color='red', label="Mejor recompensa", zorder=5)
plt.xlabel("Episodio")
plt.ylabel("Recompensa acumulada")
plt.title("Aprendizaje Fase 2")
plt.grid(True)
plt.legend()
plt.show()

Aquí se prueba visualmente el modelo entrenado: primero ejecuta 3 episodios de prueba para la Fase 1 y luego 3 episodios para la Fase 2, mostrando en pantalla los pasos y la recompensa acumulada de cada episodio.

In [ ]:
# PRUEBA DEL MODELO
probar_modelo(fase=1)    
probar_modelo(fase=2)